# 15 — 剝開最後一層：原生 OpenAI SDK 與 Anthropic (Claude) SDK 的工具呼叫

**這份要學什麼**
- 原生 OpenAI SDK 跟 Anthropic SDK 的工具呼叫格式差在哪裡
- LangChain 的 `AIMessage.tool_calls` 幫你抹平了哪些 provider 差異

> 不需要 API key。這裡用 SDK 自己的 Pydantic 型別**建構**出跟真實回應一模一樣的物件
> （不送任何網路請求），讓你看清楚原始格式長什麼樣子；有 key 的話，每段最後也留了真的
> 呼叫 API 的對照 cell。

## 這份 notebook 在做什麼——先講清楚

`02_three_agents_compare.ipynb` 說過：`create_agent` 底層是編譯出一個 LangGraph
`StateGraph`。這份 notebook 再往下剝一層——LangChain 的 `bind_tools()` /
`AIMessage.tool_calls` 這個統一格式，底層其實是在幫你把不同 provider（OpenAI、
Anthropic）的原始回應格式差異抹平。

打個比方：LangChain 的統一介面像一份「懶人包」——把重點整理好給你，好用，但幫你把細節
藏起來了；原生 SDK 的回應則像原廠說明書——比較囉唆，但你能看到最原始的每一個欄位。這份
notebook 就是讓你翻一次原廠說明書，看看懶人包底下到底藏了什麼。

**這不是要你以後改用原生 SDK 寫 agent**——`05` 的 `ToolNode`、`create_agent` 該怎麼用還是
怎麼用。這裡純粹是讓你知道「統一介面」到底幫你省了多少事，之後遇到奇怪的相容性問題時
知道去哪裡查。

## 兩邊都要先把工具描述成 JSON Schema——但欄位名稱不一樣

同一個 `get_weather(city: str)` 工具，兩家 SDK 要求的 tool 定義格式不同：

| | OpenAI | Anthropic |
|---|---|---|
| 外層 | `{"type": "function", "function": {...}}` | 沒有外層 `type` 包裝，直接攤平 |
| 參數 schema 欄位名 | `parameters` | `input_schema` |

（欄位名稱是從已安裝的 `openai` / `anthropic` SDK 型別定義 `typing.get_type_hints()`
直接讀出來驗證過的，不是憑印象寫。）

In [1]:
import json
import os

WEATHER_JSON_SCHEMA = {
    "type": "object",
    "properties": {"city": {"type": "string"}},
    "required": ["city"],
}


def get_weather(city: str) -> str:
    return f"{city}: sunny, 28C"


openai_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": WEATHER_JSON_SCHEMA,
        },
    }
]

anthropic_tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": WEATHER_JSON_SCHEMA,
    }
]

## OpenAI：`chat.completions.create` 回應長怎樣

模型想呼叫工具時，`response.choices[0].finish_reason == "tool_calls"`，工具呼叫本身在
`message.tool_calls`，每個 `function.arguments` 是**一個 JSON 字串**（不是 dict），
你要自己 `json.loads()`。下面直接用 SDK 的型別建構出這個形狀，不需要真的打 API。

In [2]:
from openai.types.chat.chat_completion import ChatCompletion, Choice
from openai.types.chat.chat_completion_message import ChatCompletionMessage
from openai.types.chat.chat_completion_message_function_tool_call import (
    ChatCompletionMessageFunctionToolCall,
    Function,
)

fake_openai_response = ChatCompletion(
    id="chatcmpl_demo",
    object="chat.completion",
    created=0,
    model="gpt-4.1-mini",
    choices=[
        Choice(
            index=0,
            finish_reason="tool_calls",
            message=ChatCompletionMessage(
                role="assistant",
                content=None,
                tool_calls=[
                    ChatCompletionMessageFunctionToolCall(
                        id="call_1",
                        type="function",
                        function=Function(name="get_weather", arguments='{"city": "Taipei"}'),
                    )
                ],
            ),
        )
    ],
)

message = fake_openai_response.choices[0].message
print("finish_reason:", fake_openai_response.choices[0].finish_reason)
print("tool_calls:", message.tool_calls)

finish_reason: tool_calls
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_1', function=Function(arguments='{"city": "Taipei"}', name='get_weather'), type='function')]


## 手動解析、執行工具、組回下一輪的訊息

沒有 `ToolNode` 幫你做這件事，OpenAI 的做法是自己 parse `arguments`、執行函式，把結果包成
一則 `role="tool"` 的訊息（要帶 `tool_call_id` 對應回去），加進訊息列表裡再呼叫一次。

In [3]:
tool_call = message.tool_calls[0]
args = json.loads(tool_call.function.arguments)  # 手動 json.loads，LangChain 的 ToolNode 幫你做了這步
result = get_weather(**args)

next_messages = [
    {"role": "user", "content": "台北天氣如何？"},
    {"role": "assistant", "content": None, "tool_calls": [tool_call.model_dump()]},
    {"role": "tool", "tool_call_id": tool_call.id, "content": result},
]
for m in next_messages:
    print(m)

{'role': 'user', 'content': '台北天氣如何？'}
{'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call_1', 'function': {'arguments': '{"city": "Taipei"}', 'name': 'get_weather'}, 'type': 'function'}]}
{'role': 'tool', 'tool_call_id': 'call_1', 'content': 'Taipei: sunny, 28C'}


## Anthropic (Claude)：`messages.create` 回應長怎樣

結構不一樣：沒有 `tool_calls` 這個欄位，工具呼叫是 `content` 列表裡的一個
`ToolUseBlock`（`type="tool_use"`），跟純文字的 `TextBlock` 混在同一個列表裡；模型要停下來
等工具結果時 `stop_reason == "tool_use"`。`ToolUseBlock.input` 已經是 dict，**不用**
自己 `json.loads()`——這是跟 OpenAI 格式最直接的差異之一。

In [4]:
from anthropic.types import Message, ToolUseBlock, Usage

fake_anthropic_response = Message(
    id="msg_demo",
    type="message",
    role="assistant",
    model="claude-sonnet-5",
    content=[ToolUseBlock(type="tool_use", id="toolu_1", name="get_weather", input={"city": "Taipei"})],
    stop_reason="tool_use",
    stop_sequence=None,
    usage=Usage(input_tokens=10, output_tokens=5),
)

print("stop_reason:", fake_anthropic_response.stop_reason)
print("content blocks:", fake_anthropic_response.content)

stop_reason: tool_use
content blocks: [ToolUseBlock(id='toolu_1', caller=None, input={'city': 'Taipei'}, name='get_weather', type='tool_use', toolset_name=None)]


In [5]:
tool_use_block = next(b for b in fake_anthropic_response.content if b.type == "tool_use")
result = get_weather(**tool_use_block.input)  # input 已經是 dict，不用 json.loads

next_messages_anthropic = [
    {"role": "user", "content": "台北天氣如何？"},
    {"role": "assistant", "content": [tool_use_block.model_dump()]},
    {
        "role": "user",  # 注意：Anthropic 的工具結果是包在下一則 "user" 訊息裡，不是獨立的 role
        "content": [{"type": "tool_result", "tool_use_id": tool_use_block.id, "content": result}],
    },
]
for m in next_messages_anthropic:
    print(m)

{'role': 'user', 'content': '台北天氣如何？'}
{'role': 'assistant', 'content': [{'id': 'toolu_1', 'caller': None, 'input': {'city': 'Taipei'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]}
{'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_1', 'content': 'Taipei: sunny, 28C'}]}


## 三種格式並排：這正是 LangChain 幫你抹平的東西

兩條路都能達到「模型呼叫 `get_weather` 查台北天氣」這個結果，但中間發生的事完全不同：

```
                         「台北天氣如何？」
                                │
              ┌─────────────────┴─────────────────┐
              ▼                                   ▼
      統一介面（02、05 走這條）              直接呼叫原生 SDK（這份 notebook 拆給你看）
              │                                   │
   model.bind_tools(tools).invoke(...)   openai.chat.completions.create(tools=...)
              │                          anthropic.messages.create(tools=...)
              ▼                                   ▼
   AIMessage.tool_calls[]                OpenAI: message.tool_calls[].function
   （不管哪家模型，dict 格式都一樣）        .arguments 是 JSON 字串，要自己 json.loads
              │                          Anthropic: content[] 裡的 ToolUseBlock，
              ▼                          input 已經是 dict
   ToolNode 自動組出 ToolMessage                    │
   （05 用過，不用手動處理）                          ▼
                                          自己 json.loads、自己組
                                          role="tool" 或 tool_result 訊息
```

| | 工具呼叫在哪 | 參數格式 | 結果怎麼送回去 |
|---|---|---|---|
| **OpenAI 原生** | `message.tool_calls[].function` | `arguments` 是 JSON 字串，要 `json.loads` | 新增一則 `role="tool"` 訊息 |
| **Anthropic 原生** | `message.content[]` 裡的 `ToolUseBlock` | `input` 已經是 dict | 包成 `user` 訊息裡的 `tool_result` 內容區塊 |
| **LangChain / LangGraph** | `AIMessage.tool_calls[]` | 統一是 dict（`02`、`05` 用的格式） | `ToolNode` 自動產生 `ToolMessage`，不用手動組 |

`05_langgraph_tools_and_agent.ipynb` 裡不管你接的是 OpenAI 還是 Anthropic 模型，
`tool_calls` 的格式、`ToolNode` 的行為都完全一樣——這就是 `bind_tools()` /
`AIMessage.tool_calls` 這層抽象的價值：換 provider 不用重寫工具呼叫迴圈。

## 如果你有 API key：接真的 API 各跑一次
分別檢查 `OPENAI_API_KEY` 跟 `ANTHROPIC_API_KEY`，有的話就真的打一次對應的 API。

In [6]:
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI

    real_response = OpenAI().chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": "台北天氣如何？"}],
        tools=openai_tools,
    )
    print(real_response.choices[0].message.tool_calls)
else:
    print("尚未設定 OPENAI_API_KEY，跳過真的呼叫（上面用型別建構的版本已經展示了完整形狀）。")

尚未設定 OPENAI_API_KEY，跳過真的呼叫（上面用型別建構的版本已經展示了完整形狀）。


In [7]:
if os.environ.get("ANTHROPIC_API_KEY"):
    from anthropic import Anthropic

    real_response = Anthropic().messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": "台北天氣如何？"}],
        tools=anthropic_tools,
    )
    print(real_response.content)
else:
    print("尚未設定 ANTHROPIC_API_KEY，跳過真的呼叫（上面用型別建構的版本已經展示了完整形狀）。")

尚未設定 ANTHROPIC_API_KEY，跳過真的呼叫（上面用型別建構的版本已經展示了完整形狀）。


## 小結
- 原生 SDK 各自有各自的工具呼叫格式，`arguments` 是字串還是 dict、結果要包成什麼 role，
  兩家完全不同
- LangChain 的 `AIMessage.tool_calls` 是一層抹平差異的正規化格式，這也是為什麼 `05`
  寫的 `ToolNode` 邏輯換模型 provider 不用改
- 什麼時候該跳過 LangChain、直接用原生 SDK：只用單一 provider、想要最少的抽象層、或需要
  用到 LangChain 還沒包裝的 provider 專屬功能時

下一份：`16_capstone_it_ticket_agent.ipynb`，把整個系列學過的東西——工具、MCP、記憶、
human-in-the-loop、streaming、多 agent——組成一個完整的實際應用場景。